## Steps for Gold
- Rename timestamp → settlement_period
- Drop validation columns
- Join to carbon intensity Gold
- No rolling averages needed   


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Load the silver table

gold_weather = spark.read.table("silver_weather")
display(gold_weather.limit(10))

In [0]:
    # Rename time column

    gold_weather = gold_weather.withColumnRenamed("timestamp", "settlement_period")
    

In [0]:
# Drop validation columns

validation_columns = ['wind_valid', 'temp_valid', 'rad_valid', 'has_nulls']

gold_weather = gold_weather.drop(*validation_columns)



In [0]:
# Join to carbon intensity Gold (regional edition)

carbon_intensity = spark.read.table("gold_carbon_intensity_regional")

gold_carbon_weather = carbon_intensity.join(gold_weather, on=['region_id', 'settlement_period'], how='inner')

display(gold_carbon_weather.limit(10))


In [0]:
# Get the rolling average of the temperature (other weather features don't really make sense)

lag_window = Window.partitionBy("region_id").orderBy("settlement_period")

# t-1 lag (30 mins)

gold_carbon_weather = gold_carbon_weather.withColumn("temp_roll_avg_7day", F.lag("temperature").over(lag_window))



In [0]:
display(gold_carbon_weather.limit(20))

In [0]:
display(gold_carbon_weather.describe())